# Week 3, day 2 — Extra practice 03 SOLUTIONS: unstack   (L04)

Executed in the lab image (pandas 3.0.5) against the real
`../data/orders_long.csv`. Every quoted number is what it actually printed.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Extra practice 03 — unstack. Run this once.
import pandas as pd

orders = pd.read_csv("../data/orders_long.csv")

by_cat_mode = orders.groupby(["Category", "ShipMode"])["Sales"].sum().round(2)
print(by_cat_mode.to_string())

### Question 1

`(3, 3)`, `9` cells, `9` combinations present, **`0`** `NaN`.

Every category ships by every mode, so the grid is full and unstacking is
lossless. Worth confirming before you rely on the shape.

In [ ]:
wide = by_cat_mode.unstack()
print(wide.to_string())
print()
print("shape:", wide.shape, "| cells:", wide.size)
print("combinations present:", len(by_cat_mode))
print("NaN cells:", int(wide.isna().sum().sum()))

### Question 2

The two layouts are transposes of each other — categories down vs ship modes down.

Same nine numbers, and the one to publish is the one whose rows are the
thing you are comparing. Three categories read more naturally down the side
than three ship modes, but that is a judgement about the audience rather
than about the data.

In [ ]:
print("unstack('ShipMode') -- categories down:")
print(by_cat_mode.unstack("ShipMode").to_string())
print()
print("unstack('Category') -- ship modes down:")
print(by_cat_mode.unstack("Category").to_string())

### Question 3

`1` `NaN` cell -> **`('Northwest Territories', 'Delivery Truck')`**.

One combination out of 24 never happened: no order was ever delivered by
truck to the Northwest Territories.

That is a plausible fact about the world rather than a data problem, and it
is only visible because unstacking made the grid rectangular. In the long
format it was an absent row.

In [ ]:
counts = orders.groupby(["Region", "ShipMode"])["Sales"].count()
wide = counts.unstack()
print(wide.to_string())
print()
print("NaN cells:", int(wide.isna().sum().sum()))
missing = wide.isna().stack()
print("missing pairs:", missing[missing].index.tolist())

### Question 4

`int64` before unstacking -> **`float64`** after. -> `fill_value=0` keeps it `int64`.

This is the dtype promotion worksheet 03 Q5 could not demonstrate, because
its column was already float.

A count is a genuine integer. One missing combination forces the whole
table to float, so every count prints with a `.0` and any code checking for
an integer dtype now takes the wrong branch. Nothing warned you.

In [ ]:
counts = orders.groupby(["Region", "ShipMode"])["Sales"].count()
print("before unstack:", counts.dtype)
print("after unstack: ", counts.unstack().dtypes.unique())
print()
print("with fill_value=0:", counts.unstack(fill_value=0).dtypes.unique())

### Question 5

Both sums are `1093`, matching the row count.

`sum()` skips `NaN` and adding zero changes nothing, so the total is the
same either way — the same result as worksheet 03 Q7, where the totals
agreed and the *means* did not.

In [ ]:
counts = orders.groupby(["Region", "ShipMode"])["Sales"].count()
plain = counts.unstack()
filled = counts.unstack(fill_value=0)
print("sum with NaN:", int(plain.sum().sum()))
print("sum with 0:  ", int(filled.sum().sum()))
print("rows in file:", len(orders))

### Question 6

Counting the negative cells **without** `.dropna()` gives `24`; with it, **`2`** — `Nunavut/Furniture -142.85` and `Quebec/Furniture -9.84`.

`m[m < 0]` does not remove the cells that fail the test, it blanks them.
So the frame still has all 24 cells and 22 of them are `NaN` — and `stack()`
in pandas 3 keeps `NaN` rows rather than dropping them.

That is worksheet 04 Q6 arriving somewhere you were not expecting it. The
naive count is twelve times too high and looks perfectly reasonable.

The habit worth taking away: after any boolean mask on a frame, `.stack()`
needs `.dropna()` unless you specifically want the blanks.

In [ ]:
m = orders.groupby(["Region", "Category"])["Profit"].mean().unstack().round(2)
print(m.to_string())
print()
naive = m[m < 0].stack()
real = m[m < 0].stack().dropna()
print("count without dropna():", len(naive), "<- wrong, most are NaN")
print("count with dropna():   ", len(real))
print()
print(real.to_string())

### Question 7

The one missing pair is `('Northwest Territories', 'Delivery Truck')`.

In a **count** table, `fill_value=0` is true — they placed zero such orders.

In a **mean profit** table it would be false. Zero asserts they broke even,
and there is no order to have broken even on. The honest cell is `NaN`.

Same data, same missing combination, and the correct fill differs by what
the cells measure. That is why `fill_value` cannot be a habit.

In [ ]:
counts = orders.groupby(["Region", "ShipMode"])["Sales"].count().unstack()
missing = counts.isna().stack()
print("pairs with no orders at all:", missing[missing].index.tolist())
print()
print("In a COUNT table, 0 is true: they placed zero orders.")
print("In a MEAN PROFIT table, 0 would assert they broke even -- but there")
print("is no order to have broken even on. NaN is the honest answer.")

### Question 8

`unstack()` on a single-level index -> **raises** `ValueError: index must be a MultiIndex to unstack, <class 'pandas.Index'> was passed`.

There is no level to move. `unstack` needs at least two, which is why it
almost always follows a `groupby` on two or more columns.

A clear message that names the type it got. Compare with worksheet 03 Q10,
where the index *was* a MultiIndex and the named level simply did not exist.

In [ ]:
flat = orders.groupby("Region")["Sales"].sum()
print("index type:", type(flat.index).__name__, "| levels:", flat.index.nlevels)
print(flat.unstack())